In [1]:
'''
*Version: 1.0 Published: 2024/02/14* Source: [NASA POWER](https://power.larc.nasa.gov/)
POWER Remotely Connect to, Slice, and Download from a POWER Zarr via Python
This is an overview of the process to connect to and download from a POWER Zarr-formatted ARD via Python.
'''

import os
import fsspec

import pandas as pd
import xarray as xr

from datetime import datetime

filepath = 'https://nasa-power.s3.us-west-2.amazonaws.com/syn1deg/temporal/power_syn1deg_daily_temporal_lst.zarr'
filepath_mapped = fsspec.get_mapper(filepath)
ds = xr.open_zarr(filepath_mapped, consolidated=True)

# List of variables to include
vars_to_keep = [
    "ALLSKY_KT", "ALLSKY_SFC_LW_DWN", "ALLSKY_SFC_LW_UP", "ALLSKY_SFC_PAR_TOT",
    "ALLSKY_SFC_SW_DIFF", "ALLSKY_SFC_SW_DNI", "ALLSKY_SFC_SW_DWN", "ALLSKY_SFC_SW_UP",
    "ALLSKY_SFC_UV_INDEX", "ALLSKY_SFC_UVA", "ALLSKY_SFC_UVB", "ALLSKY_SRF_ALB",
    "AOD_55", "AOD_55_ADJ", "CLOUD_AMT", "CLOUD_AMT_DAY", "CLOUD_AMT_NIGHT",
    "CLOUD_OD", "CLRSKY_DAYS", "CLRSKY_KT", "CLRSKY_SFC_LW_DWN", "CLRSKY_SFC_LW_UP",
    "CLRSKY_SFC_PAR_TOT", "CLRSKY_SFC_SW_DIFF", "CLRSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_UP",
    "CLRSKY_SRF_ALB", "MIDDAY_INSOL", "TOA_SW_DNI", "TOA_SW_DWN"
    
]

# Slice dataset by variables
ds_subset = ds[vars_to_keep]

# Then slice by time and region
ds_sliced = ds_subset.sel(
    time=pd.date_range(datetime(2018, 1, 1), datetime(2024, 12, 31), freq='1D'),
    lat=slice(45.53, 49),
    lon=slice(-124.77, -116.92)
).load()

output = r'' # if none the location of the script is where the files will be outputted.

# export as CSV
df_region = ds_sliced.to_dataframe()
df_region.to_csv(os.path.join(output, "climatology_radiation.csv"))

In [2]:
"""
NASA POWER MERRA-2 Meteorology Data Extraction from Zarr
Extracts meteorological parameters for specified time range and spatial extent
Time range: 2018-01-01 to 2024-12-31
Spatial extent: Lat 45.53-49°N, Lon -124.77 to -116.92°W
"""

import os
import fsspec
import pandas as pd
import xarray as xr
from datetime import datetime

# MERRA-2 Zarr filepath on NASA POWER AWS S3
# Note: The exact filepath may vary. Common patterns:
# - power_merra2_daily_temporal_lst.zarr for daily data
# - Check NASA POWER documentation for the current zarr location
filepath = 'https://nasa-power.s3.us-west-2.amazonaws.com/merra2/temporal/power_merra2_daily_temporal_lst.zarr'

print("Connecting to MERRA-2 Zarr datastore...")
try:
    filepath_mapped = fsspec.get_mapper(filepath)
    ds = xr.open_zarr(filepath_mapped, consolidated=True)
    print("✓ Successfully connected to MERRA-2 datastore")
except Exception as e:
    print(f"✗ Error connecting to Zarr datastore: {e}")
    print("\nTrying alternative filepath...")
    # Alternative filepath structure
    filepath = 'https://nasa-power.s3.us-west-2.amazonaws.com/power_merra2_daily.zarr'
    filepath_mapped = fsspec.get_mapper(filepath)
    ds = xr.open_zarr(filepath_mapped, consolidated=True)
    print("✓ Successfully connected using alternative path")

# Display available variables
print("\n" + "="*70)
print("AVAILABLE VARIABLES IN MERRA-2 DATASET:")
print("="*70)
for var in ds.data_vars:
    print(f"  - {var}")
print("="*70)

# Common MERRA-2 meteorology parameters
# You can modify this list based on the available variables shown above
vars_to_keep = [
    # Temperature parameters
    "T2M",              # Temperature at 2 Meters (°C)
    "T2MDEW",           # Dew/Frost Point at 2 Meters (°C)
    "T2MWET",           # Wet Bulb Temperature at 2 Meters (°C)
    "T2M_MAX",          # Maximum Temperature at 2 Meters (°C)
    "T2M_MIN",          # Minimum Temperature at 2 Meters (°C)
    "T2M_RANGE",        # Temperature Range at 2 Meters (°C)
    "TS",               # Earth Skin Temperature (°C)
    
    # Humidity parameters
    "RH2M",             # Relative Humidity at 2 Meters (%)
    "QV2M",             # Specific Humidity at 2 Meters (g/kg)
    
    # Precipitation parameters
    "PRECTOTCORR",      # Precipitation Corrected (mm/day)
    "PRECTOTCORR_SUM",  # Precipitation Corrected Sum (mm)
    
    # Wind parameters
    "WS2M",             # Wind Speed at 2 Meters (m/s)
    "WS10M",            # Wind Speed at 10 Meters (m/s)
    "WS50M",            # Wind Speed at 50 Meters (m/s)
    "WD2M",             # Wind Direction at 2 Meters (Degrees)
    "WD10M",            # Wind Direction at 10 Meters (Degrees)
    "WD50M",            # Wind Direction at 50 Meters (Degrees)
    
    # Pressure parameters
    "PS",               # Surface Pressure (kPa)
    "PSC",              # Corrected Atmospheric Pressure (kPa)
    
    # Additional meteorological parameters
    "GWETROOT",         # Root Zone Soil Wetness (1)
    "GWETTOP",          # Surface Soil Wetness (1)
    "EVPTRNS",          # Evapotranspiration Energy Flux (MJ/m²/day)
]

# Filter to only include variables that exist in the dataset
available_vars = [var for var in vars_to_keep if var in ds.data_vars]
missing_vars = [var for var in vars_to_keep if var not in ds.data_vars]

if missing_vars:
    print("\n⚠ WARNING: The following requested variables are not available:")
    for var in missing_vars:
        print(f"  - {var}")

if not available_vars:
    print("\n✗ ERROR: None of the requested variables are available in this dataset.")
    print("Please check the available variables list above and update the script.")
    exit()

print(f"\n✓ Found {len(available_vars)} available variables to extract")
print("\nExtracting variables:")
for var in available_vars:
    print(f"  - {var}")

# Slice dataset by variables
print("\nSlicing dataset by variables...")
ds_subset = ds[available_vars]

# Define time and spatial extent
time_range = pd.date_range(datetime(2018, 1, 1), datetime(2024, 12, 31), freq='1D')
lat_slice = slice(45.53, 49)
lon_slice = slice(-124.77, -116.92)

print(f"\nTime range: {time_range[0]} to {time_range[-1]} ({len(time_range)} days)")
print(f"Latitude range: {lat_slice.start}° to {lat_slice.stop}°N")
print(f"Longitude range: {lon_slice.start}° to {lon_slice.stop}°W")

# Slice by time and region
print("\nSlicing dataset by time and spatial extent...")
print("This may take several minutes depending on data size...")
ds_sliced = ds_subset.sel(
    time=time_range,
    lat=lat_slice,
    lon=lon_slice
).load()

print(f"✓ Data loaded successfully")
print(f"  Dataset size: {ds_sliced.nbytes / 1e6:.2f} MB")

# Define output directory and filename
output_dir = "nasa_power_data"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "merra2_meteorology.csv")

# Convert to DataFrame and export as CSV
print(f"\nConverting to DataFrame and exporting to CSV...")
df_region = ds_sliced.to_dataframe()

# Reset index to make time, lat, lon regular columns instead of multi-index
df_region = df_region.reset_index()

# Export to CSV
print(f"Saving to: {output_file}")
df_region.to_csv(output_file, index=False)

print("\n" + "="*70)
print("EXTRACTION COMPLETE!")
print("="*70)
print(f"✓ CSV file saved: {output_file}")
print(f"  Total rows: {len(df_region):,}")
print(f"  Total columns: {len(df_region.columns)}")
print(f"  File size: {os.path.getsize(output_file) / 1e6:.2f} MB")
print(f"\nFirst few rows of the CSV:")
print(df_region.head(10))
print(f"\nColumn names:")
print(df_region.columns.tolist())
print("\n" + "="*70)

print("\n✓ Data successfully exported to CSV!")
print(f"  Location: {os.path.abspath(output_file)}")
print("\nDone!")

Connecting to MERRA-2 Zarr datastore...
✓ Successfully connected to MERRA-2 datastore

AVAILABLE VARIABLES IN MERRA-2 DATASET:
  - CDD0
  - CDD10
  - CDD18_3
  - DISPH
  - EVLAND
  - EVPTRNS
  - FROST_DAYS
  - FRSEAICE
  - FRSNO
  - GWETPROF
  - GWETROOT
  - GWETTOP
  - HDD0
  - HDD10
  - HDD18_3
  - PBLTOP
  - PRECSNO
  - PRECSNOLAND
  - PRECTOTCORR
  - PS
  - QV10M
  - QV2M
  - RH2M
  - RHOA
  - SLP
  - SNODP
  - T10M
  - T10M_MAX
  - T10M_MIN
  - T10M_RANGE
  - T2M
  - T2MDEW
  - T2MWET
  - T2M_MAX
  - T2M_MIN
  - T2M_RANGE
  - TO3
  - TQV
  - TROPPB
  - TROPQ
  - TROPT
  - TS
  - TSOIL1
  - TSOIL2
  - TSOIL3
  - TSOIL4
  - TSOIL5
  - TSOIL6
  - TSURF
  - TS_MAX
  - TS_MIN
  - TS_RANGE
  - U10M
  - U2M
  - U50M
  - V10M
  - V2M
  - V50M
  - WD10M
  - WD2M
  - WD50M
  - WS10M
  - WS10M_MAX
  - WS10M_MIN
  - WS10M_RANGE
  - WS2M
  - WS2M_MAX
  - WS2M_MIN
  - WS2M_RANGE
  - WS50M
  - WS50M_MAX
  - WS50M_MIN
  - WS50M_RANGE
  - Z0M

⚠ WARNING: The following requested variables are not a